## Calidad, Limpieza y Preparación de Datos

En este notebook planteamos todas las decisiones de limpieza que aplicamos
al dataset original. Para cada transformación se indica qué se observó,
qué acción se tomó y qué impacto tuvo en el dataset.

In [ ]:
import pandas as pd
import json
import numpy as np

with open('../data/raw/streaming_users_dirty.json', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
n_inicial = df.shape[0]
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(df['subscription_plan'].unique())

Dataset cargado: 8160 filas, 8 columnas
<ArrowStringArray>
[ 'Estándar',    'Básico',   'Premium',       'Std',  'estandar',    'basico',
    'básico',  'Premium ',   'premium',   'Premiun',    'BASICO',  'STANDARD',
     'Basic', 'Estándar ',   'PREMIUM']
Length: 15, dtype: str


## Paso 1 - Eliminación de filas duplicadas

Se detectaron 126 filas duplicadas en la inspección inicial.
Los duplicados exactos no aportan información y se eliminan.

In [ ]:
filas_antes = df.shape[0]
df = df.drop_duplicates()
print(f"Filas antes: {filas_antes}")
print(f"Filas después: {df.shape[0]}")
print(f"Filas eliminadas: {filas_antes - df.shape[0]}")

Filas antes: 8160
Filas después: 8034
Filas eliminadas: 126


## Paso 2 - Normalización de subscription_plan

Se detectaron 15 variantes para 3 planes.
Se unifican a sus valores canónicos: Básico, Estándar, Premium.

In [ ]:
def normalizar_plan(valor):
    valor = str(valor).strip().lower()
    if valor in ['básico', 'basico', 'basic']:
        return 'Básico'
    elif valor in ['estándar', 'estandar', 'std', 'standard']:
        return 'Estándar'
    elif valor in ['premium', 'premiun']:
        return 'Premium'
    else:
        return None

df['subscription_plan'] = df['subscription_plan'].apply(normalizar_plan)
print("Valores únicos después de normalizar:")
print(df['subscription_plan'].value_counts())
print(f"Nulos generados: {df['subscription_plan'].isnull().sum()}")

Valores únicos después de normalizar:
subscription_plan
Básico      3609
Estándar    2833
Premium     1592
Name: count, dtype: int64
Nulos generados: 0


## Paso 3 - Normalización de country

Se detectaron 26 variantes para 7 países.
Se unifican abreviaturas y nombres en otros idiomas a su forma canónica.

In [6]:
def normalizar_pais(valor):
    valor = str(valor).strip().lower()
    if valor in ['argentina', 'arg']:
        return 'Argentina'
    elif valor in ['brasil', 'brazil', 'bra']:
        return 'Brasil'
    elif valor in ['chile', 'chl']:
        return 'Chile'
    elif valor in ['colombia', 'col']:
        return 'Colombia'
    elif valor in ['méxico', 'mexico', 'mex']:
        return 'México'
    elif valor in ['perú', 'peru', 'per']:
        return 'Perú'
    elif valor in ['uruguay', 'ury']:
        return 'Uruguay'
    else:
        return None

df['country'] = df['country'].apply(normalizar_pais)
print("Valores únicos después de normalizar:")
print(df['country'].value_counts())
print(f"Nulos generados: {df['country'].isnull().sum()}")

Valores únicos después de normalizar:
country
Chile        1167
Brasil       1164
México       1162
Uruguay      1146
Colombia     1145
Perú         1139
Argentina    1111
Name: count, dtype: int64
Nulos generados: 0


## Paso 4 - Normalización de favorite_genre e imputación de nulos

Se detectaron 28 variantes para 7 géneros y 240 valores nulos.
Se normalizan las variantes y se imputan los nulos con la moda.

In [7]:
def normalizar_genero(valor):
    if pd.isnull(valor):
        return None
    valor = str(valor).strip().lower()
    if valor in ['comedia', 'comedy']:
        return 'Comedia'
    elif valor in ['drama']:
        return 'Drama'
    elif valor in ['documental', 'documentary', 'doc']:
        return 'Documental'
    elif valor in ['thriller', 'thriler']:
        return 'Thriller'
    elif valor in ['romance']:
        return 'Romance'
    elif valor in ['acción', 'accion', 'action']:
        return 'Acción'
    elif valor in ['crime', 'crimen']:
        return 'Crime'
    else:
        return None

df['favorite_genre'] = df['favorite_genre'].apply(normalizar_genero)

# Imputamos los nulos con la moda
moda_genero = df['favorite_genre'].mode()[0]
df['favorite_genre'] = df['favorite_genre'].fillna(moda_genero)
print(f"Nulos imputados con moda: {moda_genero}")
print(df['favorite_genre'].value_counts())

Nulos imputados con moda: Comedia
favorite_genre
Comedia       1381
Drama         1121
Romance       1113
Documental    1111
Acción        1110
Thriller      1109
Crime         1089
Name: count, dtype: int64


## Paso 5 - Tratamiento de valores imposibles en age

Se detectaron registros con edad negativa y mayores a 100.
Estos valores son imposibles y no pueden imputarse, por lo que se eliminan.
El resto de los registros con edades válidas se conservan.

In [8]:
filas_antes = df.shape[0]
df = df[(df['age'] >= 0) & (df['age'] <= 100)]
print(f"Filas eliminadas por edad inválida: {filas_antes - df.shape[0]}")
print(f"Rango de edad resultante: {df['age'].min()} - {df['age'].max()}")

Filas eliminadas por edad inválida: 74
Rango de edad resultante: 0 - 80


## Paso 6 - Tratamiento de monthly_watch_time_mins

Se detectaron 193 valores nulos, 49 negativos y un outlier extremo (99999).
Los valores negativos son imposibles y se eliminan.
El outlier extremo se winsoriza al límite superior (k=3).
Los nulos se imputan con la mediana para no distorsionar la distribución.

In [9]:
# Eliminamos valores negativos
filas_antes = df.shape[0]
df = df[(df['monthly_watch_time_mins'].isnull()) | 
        (df['monthly_watch_time_mins'] >= 0)]
print(f"Filas eliminadas por valores negativos: {filas_antes - df.shape[0]}")

# Winsorizamos el outlier extremo con k=3
Q1 = df['monthly_watch_time_mins'].quantile(0.25)
Q3 = df['monthly_watch_time_mins'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 3 * IQR
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=limite_sup)
print(f"Límite superior winsorización: {limite_sup:.1f}")

# Imputamos nulos con la mediana
mediana = df['monthly_watch_time_mins'].median()
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(mediana)
print(f"Nulos imputados con mediana: {mediana:.1f} minutos")
print(f"Nulos restantes: {df['monthly_watch_time_mins'].isnull().sum()}")

Filas eliminadas por valores negativos: 49
Límite superior winsorización: 2704.7
Nulos imputados con mediana: 760.8 minutos
Nulos restantes: 0


## Paso 7 - Tratamiento de last_login_date

La columna está como texto y contiene fechas inválidas como "2026-15-03".
Se convierte a formato fecha. Las fechas inválidas quedan como NaN
y se imputan con la mediana de fechas válidas.

In [10]:
# Convertimos a fecha, los inválidos quedan como NaT
df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')

# Imputamos los nulos con la mediana de fechas válidas
mediana_fecha = df['last_login_date'].dropna().quantile(0.5)
df['last_login_date'] = df['last_login_date'].fillna(mediana_fecha)
print(f"Fecha mediana usada para imputación: {mediana_fecha}")
print(f"Nulos restantes: {df['last_login_date'].isnull().sum()}")
print(f"Rango de fechas: {df['last_login_date'].min()} - {df['last_login_date'].max()}")

Fecha mediana usada para imputación: 2022-02-13 00:00:00
Nulos restantes: 0
Rango de fechas: 2018-01-01 00:00:00 - 2029-01-01 00:00:00


## Paso 8 - Tratamiento de customer_support_tickets

Se detectaron 29 registros con valor -1 (imposible) que se eliminan.
Los outliers extremos (99, 150) se winsorizan al límite superior (k=3)
en lugar de eliminarlos, para conservar esos registros.

In [11]:
# Eliminamos valores negativos (imposibles)
filas_antes = df.shape[0]
df = df[df['customer_support_tickets'] >= 0]
print(f"Filas eliminadas por tickets negativos: {filas_antes - df.shape[0]}")

# Winsorizamos outliers extremos con k=3
Q1 = df['customer_support_tickets'].quantile(0.25)
Q3 = df['customer_support_tickets'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 3 * IQR
df['customer_support_tickets'] = df['customer_support_tickets'].clip(upper=limite_sup)
print(f"Límite superior winsorización: {limite_sup:.1f}")
print(f"Distribución resultante:")
print(df['customer_support_tickets'].value_counts().sort_index())

Filas eliminadas por tickets negativos: 27
Límite superior winsorización: 4.0
Distribución resultante:
customer_support_tickets
0    3519
1    2789
2    1148
3     277
4     151
Name: count, dtype: int64


## Resumen final del dataset limpio

In [12]:
print(f"Filas iniciales: {n_inicial}")
print(f"Filas finales: {df.shape[0]}")
print(f"Retención: {df.shape[0]/n_inicial*100:.1f}%")
print()
print("Nulos restantes:")
print(df.isnull().sum())
df.head()

Filas iniciales: 8160
Filas finales: 7884
Retención: 96.6%

Nulos restantes:
user_id                     0
age                         0
subscription_plan           0
monthly_watch_time_mins     0
country                     0
favorite_genre              0
last_login_date             0
customer_support_tickets    0
dtype: int64


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,4
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [13]:
#df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Dataset limpio guardado en data/processed/streaming_users_clean.csv")

Dataset limpio guardado en data/processed/streaming_users_clean.csv


In [14]:
import csv

log = [
    ["Paso", "Descripción", "Filas", "Nulos", "Retención (%)"],
    [1, "Dataset original", 8160, 753, "100.0%"],
    [2, "Eliminación de filas duplicadas", 8034, 753, "98.5%"],
    [3, "Normalización de subscription_plan", 8034, 0, "98.5%"],
    [4, "Normalización de country", 8034, 0, "98.5%"],
    [5, "Normalización e imputación de favorite_genre con moda", 8034, 0, "98.5%"],
    [6, "Eliminación de edades imposibles", 7960, 0, "97.5%"],
    [7, "Winsorización e imputación de monthly_watch_time_mins", 7911, 0, "96.9%"],
    [8, "Conversión e imputación de last_login_date con mediana", 7911, 0, "96.9%"],
    [9, "Eliminación de tickets negativos y winsorización de outliers", 7884, 0, "96.6%"],
]

with open('../logs/pipeline_log.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(log)

print("Log ETL guardado correctamente.")
print(f"Filas finales: 7884 | Retención: 96.6%")

Log ETL guardado correctamente.
Filas finales: 7884 | Retención: 96.6%
